# Advanced Bucket Access

This notebook explains the bucket-facing helper APIs in a more explicit way than the quick-start.

What you will learn:
- how to list products directly from the public Maya4 bucket
- how to parse a product filename into structured metadata
- how to download only metadata and only the chunks you need
- how to inspect the cached Zarr arrays locally


## Step 1: Discover the Catalog

`list_base_files_in_bucket()` gives you the product names available at the bucket root.

We immediately pass each filename through `parse_product_filename()` so the encoded information becomes tabular and readable.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from maya4 import DEFAULT_BUCKET_ID, list_base_files_in_bucket, parse_product_filename

product_files = sorted(
    file_name
    for file_name in list_base_files_in_bucket(DEFAULT_BUCKET_ID, relative_path=True)
    if file_name.endswith('.zarr')
)

rows = []
for file_name in product_files:
    meta = parse_product_filename(file_name)
    rows.append(
        {
            'filename': file_name,
            'satellite': meta['satellite'].upper(),
            'acquisition_date': meta['acquisition_date'],
            'year': meta['acquisition_date'].year,
            'stripmap_mode': meta['stripmap_mode'],
            'polarization': meta['polarization'].upper(),
        }
    )

catalog = pd.DataFrame(rows).sort_values(['stripmap_mode', 'filename']).reset_index(drop=True)
print('products in bucket:', len(catalog))
print(catalog.head(10).to_string(index=False))


products in bucket: 32
                                                          filename satellite    acquisition_date  year  stripmap_mode polarization
s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr       S1C 2025-03-28 05:28:10  2025              1           VV
s1c-s1-raw-s-vv-20250417t025744-20250417t025817-001927-003c59.zarr       S1C 2025-04-17 02:57:44  2025              1           VV
s1c-s1-raw-s-vv-20250417t025809-20250417t025828-001927-003c59.zarr       S1C 2025-04-17 02:58:09  2025              1           VV
s1c-s1-raw-s-vv-20250424t170801-20250424t170839-002038-0042df.zarr       S1C 2025-04-24 17:08:01  2025              1           VV
s1c-s2-raw-s-vv-20250331t205042-20250331t205107-001690-002d5c.zarr       S1C 2025-03-31 20:50:42  2025              2           VV
s1c-s2-raw-s-vv-20250403t062508-20250403t062540-001725-002fa3.zarr       S1C 2025-04-03 06:25:08  2025              2           VV
s1c-s2-raw-s-vv-20250405t060506-20250405t060539-001754-00317

## Step 2: Summarize What the Filenames Tell Us

Before downloading anything, we can already inspect how the bucket is distributed across stripmap modes and polarization.

This is often enough to decide whether your filter is too broad or too narrow.


In [2]:
mode_summary = catalog.groupby('stripmap_mode').size().rename('products')
pol_summary = catalog.groupby('polarization').size().rename('products')

print('Products per stripmap mode:')
print(mode_summary.to_string())
print()
print('Products per polarization:')
print(pol_summary.to_string())


Products per stripmap mode:
stripmap_mode
1     4
2     6
3     1
4    10
5     5
6     6

Products per polarization:
polarization
VV    32


## Step 3: Pick One Product and Explain the Plan

To keep the example cheap, we select the first product in sorted order and fetch only:
- root metadata
- metadata for the `rcmc` and `az` levels
- the chunk at `(y=0, x=0)` for those arrays

This is the bucket-native pattern you want when you are inspecting data rather than mirroring the whole archive.


In [3]:
product = catalog.iloc[0]['filename']
product_meta = parse_product_filename(product)

print('selected product:', product)
print('satellite:', product_meta['satellite'].upper())
print('acquisition_date:', product_meta['acquisition_date'])
print('stripmap_mode:', product_meta['stripmap_mode'])
print('polarization:', product_meta['polarization'].upper())
print('local_cache_target:', (Path('data') / product).as_posix())


selected product: s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr
satellite: S1C
acquisition_date: 2025-03-28 05:28:10
stripmap_mode: 1
polarization: VV
local_cache_target: data/s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr


## Step 4: Download Only the Metadata and One Chunk Per Level

`download_metadata_from_product()` creates the local folder structure and fetches the metadata needed to open the Zarr groups.

`fetch_chunk_from_bucket_zarr()` then fetches a specific chunk lazily. This is the key idea behind efficient remote access.


In [4]:
import zarr
from maya4 import download_metadata_from_product, fetch_chunk_from_bucket_zarr

DATA_DIR = REPO_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

_ = download_metadata_from_product(
    zfile_name=product,
    local_dir=DATA_DIR,
    bucket_id=DEFAULT_BUCKET_ID,
    levels=['rcmc', 'az'],
    show_progress=False,
)

for level in ['rcmc', 'az']:
    fetch_chunk_from_bucket_zarr(
        level=level,
        y=0,
        x=0,
        local_dir=DATA_DIR,
        bucket_id=DEFAULT_BUCKET_ID,
        zarr_archive=product,
        show_progress=False,
    )

print('cached files are now available under:', (Path('data') / product).as_posix())


cached files are now available under: data/s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr


## Step 5: Open the Local Arrays and Inspect Them

Now that the metadata and one chunk exist locally, `zarr.open()` can inspect the arrays just like regular local Zarr data.

The shape and chunk size tell you how the product is tiled. The sample value proves the requested chunk is really present.


In [5]:
rcmc = zarr.open(DATA_DIR / product / 'rcmc', mode='r')
az = zarr.open(DATA_DIR / product / 'az', mode='r')

print('rcmc shape:', rcmc.shape, 'chunks:', rcmc.chunks, 'dtype:', rcmc.dtype)
print('az shape:', az.shape, 'chunks:', az.chunks, 'dtype:', az.dtype)
print('rcmc[0, 0]:', complex(rcmc[0, 0]))
print('az[0, 0]:', complex(az[0, 0]))


rcmc shape: (44904, 25484) chunks: (256, 256) dtype: complex128
az shape: (44904, 25484) chunks: (256, 256) dtype: complex128
rcmc[0, 0]: (5112.9243126413685-17.965251846144064j)
az[0, 0]: (2701.2872411565663+417.5754265061424j)


## Takeaways

This notebook showed the bucket workflow in layers:
1. discover files
2. parse metadata from names
3. fetch only the metadata you need
4. fetch only the chunks you care about

That is the right mental model for efficient Maya4 inspection and debugging.
